In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd


TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"

INITIAL_PARAMS_PATH = INPUT_DIR / "initial_exponential_params.csv"
STAGE_PARAMS_PATH = INPUT_DIR / "stage_grw_params.csv"
FIT_MANIFEST_PATH = INPUT_DIR / "fit_manifest.json"

TRAJECTORIES_PATH = OUTPUT_DIR / "trajectories.npy"
LOG_TRAJECTORIES_PATH = OUTPUT_DIR / "log_trajectories.npy"
SUMMARY_PATH = OUTPUT_DIR / "simulation_summary.csv"
MANIFEST_PATH = OUTPUT_DIR / "simulation_manifest.json"

MODEL_NAME = "Stagewise AR(1)-GRW"
MODEL_TAG = "stagewise_ar1_grw"

N = 10_000
Y = 20
SEED = 63

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
initial_params = pd.read_csv(INITIAL_PARAMS_PATH)
stage_params = pd.read_csv(STAGE_PARAMS_PATH)

EPS = 0.49

required_initial = {"scale_alpha"}
required_stage = {"stage","start","end","ar_intercept","beta","lognormal_mu","lognormal_sigma"}

missing_initial = sorted(required_initial - set(initial_params.columns))
missing_stage = sorted(required_stage - set(stage_params.columns))

assert not missing_initial, f"Missing initial: {missing_initial}"
assert not missing_stage, f"Missing stage: {missing_stage}"
assert len(initial_params) == 1, "Expected exactly one initial parameter row"
assert stage_params["ar_intercept"].eq(0).all()

alpha0 = float(initial_params.loc[0, "scale_alpha"])

print(f"EPS: {EPS}")
print(f"Initial exponential scale: {alpha0:.6f}")
display(stage_params[["stage","start","end","beta","lognormal_mu","lognormal_sigma"]])

EPS: 0.49
Initial exponential scale: 4.318336


,stage,start,end,beta,lognormal_mu,lognormal_sigma
0,years_1_4,1,4,0.451436,0.857612,0.983639
1,years_5_7,5,7,0.536658,0.742392,0.911151
2,years_8_20,8,20,0.600853,0.531023,0.903748


In [3]:

def stage_row_for_destination_year(destination_year, params):
    match = params.loc[params["start"].le(destination_year) & params["end"].ge(destination_year)]
    assert len(match) == 1
    return match.iloc[0]

stage_lookup = {destination_year: stage_row_for_destination_year(destination_year,stage_params) for destination_year in range(1, Y+1)}

lookup_table = pd.DataFrame([
    {"destination_year": year,
    "stage": row["stage"],
    "beta": row["beta"],
    "lognormal_mu": row["lognormal_mu"],
    "lognormal_sigma": row["lognormal_sigma"]} for year, row in stage_lookup.items()])

display(lookup_table)

,destination_year,stage,beta,lognormal_mu,lognormal_sigma
0,1,years_1_4,0.451436,0.857612,0.983639
1,2,years_1_4,0.451436,0.857612,0.983639
2,3,years_1_4,0.451436,0.857612,0.983639
3,4,years_1_4,0.451436,0.857612,0.983639
4,5,years_5_7,0.536658,0.742392,0.911151
5,6,years_5_7,0.536658,0.742392,0.911151
6,7,years_5_7,0.536658,0.742392,0.911151
7,8,years_8_20,0.600853,0.531023,0.903748
8,9,years_8_20,0.600853,0.531023,0.903748
9,10,years_8_20,0.600853,0.531023,0.903748


In [4]:

def simulate_stagewise_ar1_grw(alpha0,stage_lookup,n=N,y_max=Y,eps=EPS,seed=SEED,):
    rng = np.random.default_rng(seed)

    trajectories = np.zeros((y_max + 1, n), dtype=float)
    log_trajectories = np.zeros((y_max + 1, n), dtype=float)

    q0 = rng.exponential(scale=alpha0, size=n)
    z = np.log(q0 + eps)

    trajectories[0] = q0
    log_trajectories[0] = z

    for current_year in range(y_max):
        destination_year = current_year + 1
        params = stage_lookup[destination_year]

        beta = float(params["beta"])
        mu = float(params["lognormal_mu"])
        sigma = float(params["lognormal_sigma"])

        log_gamma = rng.normal(loc=mu,scale=sigma,size=n)

        z = beta * z + log_gamma
        q_next = np.maximum(np.exp(z) - eps, 0.0)

        trajectories[destination_year] = q_next
        log_trajectories[destination_year] = z

    return trajectories, log_trajectories

In [5]:
trajectories, log_trajectories = simulate_stagewise_ar1_grw(alpha0=alpha0,stage_lookup=stage_lookup,n=N,y_max=Y,eps=EPS,seed=SEED)

assert trajectories.shape == (Y + 1, N)
assert log_trajectories.shape == (Y + 1, N)
assert np.isfinite(trajectories).all()
assert np.isfinite(log_trajectories).all()
assert (trajectories >= 0).all()

print(f"max sim'd prod: {trajectories.max():,.3f}")

max sim'd prod: 781.104


In [6]:
simulation_summary = pd.DataFrame({
    "CareerAge": np.arange(Y + 1),
    "mean_pubs_adj": trajectories.mean(axis=1),
    "median_pubs_adj": np.median(trajectories, axis=1),
    "sd_pubs_adj": trajectories.std(axis=1, ddof=0),
    "zero_fraction": (trajectories == 0).mean(axis=1),
    "minimum_pubs_adj": trajectories.min(axis=1),
    "maximum_pubs_adj": trajectories.max(axis=1),
    "mean_log_pubs_adj": log_trajectories.mean(axis=1),
    "var_log_pubs_adj": log_trajectories.var(axis=1, ddof=0)})

display(simulation_summary)

,CareerAge,mean_pubs_adj,median_pubs_adj,sd_pubs_adj,zero_fraction,minimum_pubs_adj,maximum_pubs_adj,mean_log_pubs_adj,var_log_pubs_adj
0,0,4.332366,3.028651,4.320733,0.0000,0.000387,40.697342,1.206085,0.786653
1,1,6.500247,3.640643,9.225729,0.0247,0.000000,164.044492,1.392019,1.133383
2,2,7.492897,3.958954,12.250920,0.0232,0.000000,497.530407,1.484778,1.203418
3,3,7.742226,4.044858,12.048684,0.0193,0.000000,261.490130,1.514843,1.186650
4,4,7.864093,4.118555,13.903822,0.0207,0.000000,766.389247,1.526749,1.196433
5,5,8.047237,4.146818,12.551841,0.0162,0.000000,241.649167,1.555196,1.162880
6,6,8.298034,4.412300,12.586267,0.0163,0.000000,323.183733,1.591855,1.168393
7,7,8.380240,4.474402,13.239945,0.0161,0.000000,349.798472,1.592523,1.181092
8,8,7.952864,4.037939,12.424380,0.0240,0.000000,248.983249,1.510422,1.266573
9,9,7.561111,3.750940,12.450042,0.0268,0.000000,407.159902,1.449896,1.282242


In [7]:
np.save(TRAJECTORIES_PATH, trajectories)
np.save(LOG_TRAJECTORIES_PATH, log_trajectories)
simulation_summary.to_csv(SUMMARY_PATH, index=False)